In [1]:
from redbox.graph.root import get_root_graph
from redbox.models.settings import Settings, ElasticLocalSettings
from pathlib import Path
from dotenv import load_dotenv, dotenv_values
from langchain_openai import AzureChatOpenAI
import tiktoken
from core_api.dependencies import get_parameterised_retriever, get_all_chunks_retriever, get_embedding_model
import logging
import statistics

_ = load_dotenv(Path.cwd().parent / ".env")

ENV = Settings(
    minio_host="localhost", 
    object_store="minio", 
    unstructured_host="localhost",
    worker_ingest_min_chunk_size=1_000,
    worker_ingest_max_chunk_size=10_000,
    worker_ingest_largest_chunk_size=300_000,
    elastic=ElasticLocalSettings(host="localhost"),
)

LLM = AzureChatOpenAI(
    api_key=ENV.azure_openai_api_key_4o,
    azure_endpoint=ENV.azure_openai_endpoint_4o,
    model="gpt-4o",
)

TEST_INDEX = "rag-test-index"

In [2]:
ENV.worker_ingest_min_chunk_size, ENV.worker_ingest_max_chunk_size, ENV.worker_ingest_largest_chunk_size, ENV.worker_ingest_largest_chunk_overlap

(1000, 10000, 300000, 0)

# RAG: make it good

By hook or by crook we need to make RAG half decent. Here's where we try.

My theory is that we'll need to change both ingest and retrival methods.

## Ingest

In [3]:
from functools import partial
from elasticsearch.helpers import scan, bulk

from redbox.loader.ingester import get_elasticsearch_store, get_elasticsearch_store_without_embeddings, UnstructuredLargeChunkLoader, UnstructuredTitleLoader
from redbox.loader.base import BaseRedboxFileLoader

from langchain_core.runnables import RunnableParallel, chain, RunnableLambda
from langchain.vectorstores import VectorStore

In [15]:
from collections.abc import Iterator
from datetime import UTC, datetime

import requests
import tiktoken
from langchain_core.documents import Document

from redbox.models.file import ChunkResolution, ChunkMetadata
from redbox.loader.base import BaseRedboxFileLoader

encoding = tiktoken.get_encoding("cl100k_base")

class NewUnstructuredLargeChunkLoader(BaseRedboxFileLoader):
    """Load, partition and chunk a document using local unstructured library"""

    def lazy_load(self) -> Iterator[Document]:  # <-- Does not take any arguments
        """A lazy loader that reads a file line by line.

        When you're implementing lazy load methods, you should use a generator
        to yield documents one by one.
        """

        url = f"http://{self.env.unstructured_host}:8000/general/v0/general"
        files = {
            "files": (self.file_name, self.file_bytes),
        }
        response = requests.post(
            url,
            files=files,
            data={
                "chunking_strategy": "by_title",
                "strategy": "fast",
                "combine_under_n_chars": 280_000,
                "max_characters": 300_000,
                "overlap": self.env.worker_ingest_largest_chunk_overlap,
                "overlap_all": True,
            },
        )

        if response.status_code != 200:
            raise ValueError(response.text)

        elements = response.json()

        if not elements:
            raise ValueError("Unstructured failed to extract text for this file")

        for i, raw_chunk in enumerate(elements):
            yield Document(
                page_content=raw_chunk["text"],
                metadata=ChunkMetadata(
                    index=i,
                    file_name=raw_chunk["metadata"].get("filename"),
                    page_number=raw_chunk["metadata"].get("page_number"),
                    created_datetime=datetime.now(UTC),
                    token_count=len(encoding.encode(raw_chunk["text"])),
                    chunk_resolution=ChunkResolution.largest,
                ).model_dump(),
            )

In [22]:
import backoff
import time
from requests.exceptions import HTTPError
from datetime import datetime, timedelta

class RateLimiter:
    def __init__(self, requests_per_minute):
        self.requests_per_minute = requests_per_minute
        self.reset_time = datetime.now()
        self.request_count = 0

    def check_rate_limit(self):
        now = datetime.now()
        if now >= self.reset_time + timedelta(minutes=1):
            # Reset the counter after 1 minute
            self.request_count = 0
            self.reset_time = now
        if self.request_count >= self.requests_per_minute:
            # Wait until 1 minute is over
            sleep_time = (self.reset_time + timedelta(minutes=1) - now).total_seconds()
            time.sleep(sleep_time)
            self.request_count = 0
            self.reset_time = datetime.now()

        self.request_count += 1


def build_local_document_loader(document_loader_type: type[BaseRedboxFileLoader], env: Settings):
    @chain
    def _loader(file_path: Path):
        return document_loader_type(
            file_name=file_path.name, 
            file_bytes=file_path.open("br").read(), 
            env=env
        ).lazy_load()
    return _loader


def build_add_documents_with_backoff(vectorstore: VectorStore):
    rate_limiter = RateLimiter(requests_per_minute=3_000)


    @chain
    @backoff.on_exception(
        backoff.expo, 
        HTTPError, 
        max_tries=5, 
        giveup=lambda e: e.response.status_code not in [429, 500, 503]
    )
    def _add_documents_with_backoff(documents: list[Document]) -> list[str]:
        ids: list[str] = []
        for doc in documents:
            # Check and enforce rate limit
            rate_limiter.check_rate_limit()
            
            try:
                id = vectorstore.add_documents([doc], create_index_if_not_exists=False)
                ids.append(id)
            except HTTPError as e:
                if e.response.status_code in [429, 500, 503]:
                    logging.warning(f"Retrying due to HTTPError {e.response.status_code}")
                    raise
                else:
                    raise
        return ids
    
    return _add_documents_with_backoff


def ingest_from_bytes(document_loader_type: type[BaseRedboxFileLoader], vectorstore: VectorStore, env: Settings):
    return (
        build_local_document_loader(
            document_loader_type=document_loader_type, 
            env=env
        )
        | RunnableLambda(list)
        | build_add_documents_with_backoff(vectorstore)
    )

In [23]:
def ingest_file(file_path: Path) -> str | None:
    logging.info("Ingesting file: %s", file_path.name)

    es = ENV.elasticsearch_client()
    es_index_name = TEST_INDEX

    es.indices.create(index=es_index_name, ignore=[400])

    chunk_ingest_chain = ingest_from_bytes(
        document_loader_type=UnstructuredTitleLoader,
        vectorstore=get_elasticsearch_store(es, es_index_name),
        env=ENV,
    )

    large_chunk_ingest_chain = ingest_from_bytes(
        document_loader_type=NewUnstructuredLargeChunkLoader,
        vectorstore=get_elasticsearch_store_without_embeddings(es, es_index_name),
        env=ENV,
    )

    try:
        new_ids = RunnableParallel({
            "normal": chunk_ingest_chain, 
            "largest": large_chunk_ingest_chain
        }).invoke(file_path)

        logging.info("File: %s %s chunks ingested", file_path.name, {k: len(v) for k, v in new_ids.items()})

    except Exception as e:
        logging.exception("Error while processing file [%s]", file_path.name)
        
        return f"{type(e)}: {e.args[0]}"


In [13]:
for _min, _max, _lrg in ((1_000, 10_000, 96_000), (1_000, 10_000, 300_000)):
    # f = Path("/Users/willlangdale/Downloads/DS/KAN- Kolmogorov–Arnold Networks.pdf")
    f = Path("/Users/willlangdale/Downloads/DS/Mamba- Linear-Time Sequence Modeling with Selective State Spaces.pdf")
    e = Settings(
        minio_host="localhost", 
        object_store="minio", 
        unstructured_host="localhost",
        worker_ingest_min_chunk_size=_min,
        worker_ingest_max_chunk_size=_max,
        worker_ingest_largest_chunk_size=_lrg,
        elastic=ElasticLocalSettings(host="localhost"),
    )

    # @chain
    # def title_loader(file_path: Path):
    #     return UnstructuredTitleLoader(
    #         file_name=file_path.name, 
    #         file_bytes=file_path.open("br").read(), 
    #         env=e
    #     ).lazy_load()

    @chain
    def title_loader(file_path: Path):
        return NewUnstructuredLargeChunkLoader(
            file_name=file_path.name, 
            file_bytes=file_path.open("br").read(), 
            env=e
        ).lazy_load()

    ingest_chain = (
        title_loader
        | RunnableLambda(list)
    )

    c = ingest_chain.invoke(f)

    print(
        _min, 
        _max,
        _lrg,
        len(c),
        min([ch.metadata["token_count"] for ch in c]),
        max([ch.metadata["token_count"] for ch in c]),
        statistics.mean(ch.metadata["token_count"] for ch in c),
    )

1000 10000 96000 1 40167 40167 40167
1000 10000 300000 1 40167 40167 40167


In [25]:
for f in (Path.cwd().parents[2] / "Downloads/DS").glob("*.pdf"):
    ingest_file(file_path=f)

INFO:root:Ingesting file: KAN- Kolmogorov–Arnold Networks.pdf
/var/folders/14/6nvsrw1n2ls1xncz_bvy2x8m0000gq/T/ipykernel_31075/2382112305.py:7: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(index=es_index_name, ignore=[400])
INFO:elastic_transport.transport:PUT http://localhost:9200/rag-test-index [status:400 duration:0.004s]
INFO:elastic_transport.transport:PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.177s]
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.235s]
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 200 OK"
INFO

In [21]:
for f in (Path.cwd().parents[2] / "Downloads/Giant").glob("*.txt"):
    ingest_file(file_path=f)

INFO:root:Ingesting file: warandpeace.txt
/var/folders/14/6nvsrw1n2ls1xncz_bvy2x8m0000gq/T/ipykernel_31075/2382112305.py:7: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es.indices.create(index=es_index_name, ignore=[400])
INFO:elastic_transport.transport:PUT http://localhost:9200/rag-test-index [status:400 duration:0.005s]
INFO:elastic_transport.transport:PUT http://localhost:9200/_bulk?refresh=true [status:200 duration:0.665s]
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 503 Service Unavailable"
INFO:openai._base_client:Retrying request to /embeddings in 0.968766 seconds
INFO:httpx:HTTP Request: POST https://oai-i-dot-ai-playground-sweden.openai.azure.com//openai/deployments/text-embedding-3-large/embeddings?api-version=2024-02-01 "HTTP/1.1 503 Service Unavailable"
ERROR:root:Error whi

In [24]:
# Clear index function
def clear_index(index: str, env: Settings) -> None:
    es = env.elasticsearch_client()
    documents = scan(es, index=index, query={"query": {"match_all": {}}})
    bulk_data = [
        {"_op_type": "delete", "_index": doc['_index'], "_id": doc['_id']} for doc in documents
    ]
    bulk(es, bulk_data, request_timeout=300)

clear_index(TEST_INDEX, ENV)

INFO:elastic_transport.transport:POST http://localhost:9200/rag-test-index/_search?scroll=5m [status:200 duration:0.251s]
INFO:elastic_transport.transport:POST http://localhost:9200/_search/scroll [status:200 duration:0.003s]
INFO:elastic_transport.transport:DELETE http://localhost:9200/_search/scroll [status:200 duration:0.002s]
/var/folders/14/6nvsrw1n2ls1xncz_bvy2x8m0000gq/T/ipykernel_31075/1245632407.py:8: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  bulk(es, bulk_data, request_timeout=300)
INFO:elastic_transport.transport:PUT http://localhost:9200/_bulk [status:200 duration:0.012s]


## Retrieval